0 import cleaned data and foundatal check

In [4]:
import pandas as pd
import numpy as np
import src.data_clean_notebook as preproc 

df = pd.read_json("../data/raw_credit_applications.json")
df_preproced = preproc.run_data_quality_pipeline(df)

ModuleNotFoundError: No module named 'src'

后面需要修改上面读取的代码

In [5]:
import pandas as pd
import numpy as np

# 1) Load DS audit dataset exported from Notebook 01
df = pd.read_csv("../data/credit_applications_ds_audit_v1.csv")

print("shape:", df.shape)
print("columns:", df.columns.tolist())

# 2) Minimal sanity checks
print("\nRaw gender counts:")
print(df["applicant_gender"].value_counts(dropna=False))

print("\nRaw label counts:")
print(df["decision_loan_approved"].value_counts(dropna=False))

# 3) Standardize fields (lightweight + safe)
def normalize_gender(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["m", "male", "man"]:
        return "Male"
    if s in ["f", "female", "woman"]:
        return "Female"
    if s == "":
        return np.nan
    return np.nan

def to_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["true","1","yes","y","approved","approve"]:
        return True
    if s in ["false","0","no","n","denied","deny","rejected","reject"]:
        return False
    return np.nan

df["gender_norm"] = df["applicant_gender"].apply(normalize_gender)
df["y"] = df["decision_loan_approved"].apply(to_bool)

print("\nNormalized gender counts:")
print(df["gender_norm"].value_counts(dropna=False))

print("\nNormalized label counts (y):")
print(df["y"].value_counts(dropna=False))

shape: (502, 11)
columns: ['_id', 'decision_loan_approved', 'applicant_gender', 'applicant_date_of_birth', 'fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'spend_total', 'spend_txn_count', 'spend_unique_cats']

Raw gender counts:
applicant_gender
Female    251
Male      248
NaN         3
Name: count, dtype: int64

Raw label counts:
decision_loan_approved
True     292
False    210
Name: count, dtype: int64

Normalized gender counts:
gender_norm
Female    251
Male      248
NaN         3
Name: count, dtype: int64

Normalized label counts (y):
y
True     292
False    210
Name: count, dtype: int64


## Compute approval rates by gender

In order to compute DI rate, we  drop the three Nan, it will cause the bias, but we will consider this in missing value influence.

We first compute approval/selection rates by group and then compute Disparate Impact (DI). DI is computed on rows with non-missing protected attribute and outcome, because DI is undefined otherwise. We also report group sample sizes.

In [7]:
# Prepare data for DI computation 

cols_needed = ["gender_norm", "y"]
tmp = df[cols_needed].dropna().copy()

# make sure y is numeric 0/1
# If your y is already 0/1, keep as is.
tmp["y"] = tmp["y"].astype(int)

tmp.head()

,gender_norm,y
0,Male,0
1,Male,0
2,Male,1
3,Male,1
4,Male,0


In [8]:
# Group sizes and approval rates (select bias)

group_sizes = tmp["gender_norm"].value_counts(dropna=False)
approval_rates = tmp.groupby("gender_norm")["y"].mean().sort_values(ascending=False)

print("Group sizes:\n", group_sizes)
print("\nApproval (selection) rates by group:\n", approval_rates)

Group sizes:
 gender_norm
Female    251
Male      248
Name: count, dtype: int64

Approval (selection) rates by group:
 gender_norm
Male      0.657258
Female    0.505976
Name: y, dtype: float64


In [9]:
# Disparate Impact 

# reference group = highest approval rate
ref_group = approval_rates.idxmax()
# protected group = lowest approval rate 
prot_group = approval_rates.idxmin()

ref_rate = float(approval_rates.loc[ref_group])
prot_rate = float(approval_rates.loc[prot_group])

if ref_rate == 0:
    di = float("nan")
    flag_80 = None
    print("Reference group's approval rate is 0, DI is undefined.")
else:
    di = prot_rate / ref_rate
    flag_80 = di < 0.8

print(f"Reference group: {ref_group} | rate = {ref_rate:.4f}")
print(f"Protected group: {prot_group} | rate = {prot_rate:.4f}")
print(f"Disparate Impact (DI) = {di:.4f}")
print(f"80% rule flag (DI < 0.8): {flag_80}")

Reference group: Male | rate = 0.6573
Protected group: Female | rate = 0.5060
Disparate Impact (DI) = 0.7698
80% rule flag (DI < 0.8): True


### compute the bias risk

In [10]:
# Missingness audit for gender_norm

miss_gender = df["gender_norm"].isna()
missing_rate = miss_gender.mean()

print(f"Missing gender_norm rate: {missing_rate:.4%}")
print("Counts (missing vs non-missing):")
print(miss_gender.value_counts())

# If outcome y exists for all, compare outcome distribution
if "y" in df.columns:
    print("\nOutcome distribution when gender is missing:")
    print(df.loc[miss_gender, "y"].value_counts(dropna=False))

    print("\nOutcome distribution when gender is NOT missing:")
    print(df.loc[~miss_gender, "y"].value_counts(dropna=False))

Missing gender_norm rate: 0.5976%
Counts (missing vs non-missing):
gender_norm
False    499
True       3
Name: count, dtype: int64

Outcome distribution when gender is missing:
y
True     2
False    1
Name: count, dtype: int64

Outcome distribution when gender is NOT missing:
y
True     290
False    209
Name: count, dtype: int64


**Fairness metrics (Selection rate & Disparate Impact)**

We evaluate historical decision outcomes (approval = 1) across gender groups. Group sizes are similar (Female n=251, Male n=248). The approval/selection rate is 0.657 for Male and 0.506 for Female, a gap of ~15.1 percentage points.
We then compute Disparate Impact (DI) as the ratio of the protected group’s selection rate to the reference group’s selection rate. Using the higher-rate group as reference (Male), the DI for Female is 0.770, which violates the four-fifths rule (DI < 0.8). This indicates potential disparate impact and motivates a statistical significance test and further investigation of potential proxy variables.

**Missingness audit**

DI is computed on records with non-missing protected attribute and outcome, because DI is undefined otherwise. The missing rate for gender_norm is 0.60% (3/502), which is minimal. The missing subset is too small to draw reliable conclusions about systematic missingness; therefore the DI findings are unlikely to be driven by missing-data handling.


# Statisc Test 

We test whether approval decisions are independent of gender.

H0: gender and approval are independent.

If p < 0.05, we reject H0 and conclude the difference is statistically significant.

## Chi-square

In [12]:
from scipy.stats import chi2_contingency

ct = pd.crosstab(tmp["gender_norm"], tmp["y"])
print("Contingency table (gender x approval):\n", ct)

chi2, p, dof, expected = chi2_contingency(ct)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
print("\nExpected counts under H0 (independence):\n", expected_df.round(2))

Contingency table (gender x approval):
 y              0    1
gender_norm          
Female       124  127
Male          85  163

Chi-square statistic: 11.1156
p-value: 0.000856
Degrees of freedom: 1

Expected counts under H0 (independence):
 y                 0       1
gender_norm                
Female       105.13  145.87
Male         103.87  144.13


**Chi-Square conclusion**

The chi-square test indicates that loan approval decisions are not independent of gender (χ²(1)=11.12, p=0.000856). Therefore, we reject H0 and conclude that the difference in approval outcomes across gender groups is statistically significant. The observed pattern is consistent with the fairness metrics above: the approval rate is higher for Male than for Female, supporting the presence of disparate outcomes that warrant further investigation.

## Cramer's V

We report Cramer’s V to quantify the strength of association between gender and approval.

In [13]:
import numpy as np

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))
print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.1493
